In [ ]:
import pandas as pd

def preprocess(df,region_df):
    # filtering for summer olympics
    df = df[df['Season'] == 'Summer']
    # merge with region_df
    df = df.merge(region_df, on='NOC', how='left')
    # dropping duplicates
    df.drop_duplicates(inplace=True)
    # one hot encoding medals
    df = pd.concat([df, pd.get_dummies(df['Medal'])], axis=1)
    return df

In [ ]:
import numpy as np


def fetch_medal_tally(df, year, country):
    medal_df = df.drop_duplicates(subset=['Team', 'NOC', 'Games', 'Year', 'City', 'Sport', 'Event', 'Medal'])
    flag = 0
    if year == 'Overall' and country == 'Overall':
        temp_df = medal_df
    if year == 'Overall' and country != 'Overall':
        flag = 1
        temp_df = medal_df[medal_df['region'] == country]
    if year != 'Overall' and country == 'Overall':
        temp_df = medal_df[medal_df['Year'] == int(year)]
    if year != 'Overall' and country != 'Overall':
        temp_df = medal_df[(medal_df['Year'] == year) & (medal_df['region'] == country)]

    if flag == 1:
        x = temp_df.groupby('Year').sum()[['Gold', 'Silver', 'Bronze']].sort_values('Year').reset_index()
    else:
        x = temp_df.groupby('region').sum()[['Gold', 'Silver', 'Bronze']].sort_values('Gold',
                                                                                      ascending=False).reset_index()

    x['total'] = x['Gold'] + x['Silver'] + x['Bronze']

    x['Gold'] = x['Gold'].astype('int')
    x['Silver'] = x['Silver'].astype('int')
    x['Bronze'] = x['Bronze'].astype('int')
    x['total'] = x['total'].astype('int')

    return x


def country_year_list(df):
    years = df['Year'].unique().tolist()
    years.sort()
    years.insert(0, 'Overall')

    country = np.unique(df['region'].dropna().values).tolist()
    country.sort()
    country.insert(0, 'Overall')

    return years,country

# def data_over_time(df,col):

#     nations_over_time = df.drop_duplicates(['Year', col])['Year'].value_counts().reset_index().sort_values('index')
#     nations_over_time.rename(columns={'index': 'Edition', 'Year': col}, inplace=True)
#     return nations_over_time

def data_over_time(df, col):

    temp = df.drop_duplicates(['Year', col])
    temp = temp.groupby('Year').count()[col].reset_index()

    temp.rename(columns={'Year':'Edition', col:col}, inplace=True)

    return temp

def most_successful(df, sport):
    temp_df = df.dropna(subset=['Medal'])

    if sport != 'Overall':
        temp_df = temp_df[temp_df['Sport'] == sport]

    x = temp_df['Name'].value_counts().reset_index().head(15).merge(df, left_on='index', right_on='Name', how='left')[
        ['index', 'Name_x', 'Sport', 'region']].drop_duplicates('index')
    x.rename(columns={'index': 'Name', 'Name_x': 'Medals'}, inplace=True)
    return x

def yearwise_medal_tally(df,country):
    temp_df = df.dropna(subset=['Medal'])
    temp_df.drop_duplicates(subset=['Team', 'NOC', 'Games', 'Year', 'City', 'Sport', 'Event', 'Medal'], inplace=True)

    new_df = temp_df[temp_df['region'] == country]
    final_df = new_df.groupby('Year').count()['Medal'].reset_index()

    return final_df

def country_event_heatmap(df,country):
    temp_df = df.dropna(subset=['Medal'])
    temp_df.drop_duplicates(subset=['Team', 'NOC', 'Games', 'Year', 'City', 'Sport', 'Event', 'Medal'], inplace=True)

    new_df = temp_df[temp_df['region'] == country]

    pt = new_df.pivot_table(index='Sport', columns='Year', values='Medal', aggfunc='count').fillna(0)
    return pt


# def most_successful_countrywise(df, country):
#     temp_df = df.dropna(subset=['Medal'])

#     temp_df = temp_df[temp_df['region'] == country]

#     x = temp_df['Name'].value_counts().reset_index().head(10).merge(df, left_on='index', right_on='Name', how='left')[
#         ['index', 'Name_x', 'Sport']].drop_duplicates('index')
#     x.rename(columns={'index': 'Name', 'Name_x': 'Medals'}, inplace=True)
#     return x

def most_successful_countrywise(df, country):

    temp = df[(df['region'] == country) & (df['Medal'] != 'No Medal')]

    x = temp['Name'].value_counts().reset_index()
    x.columns = ['Name','Medals']

    return x.head(10)

def weight_v_height(df,sport):
    athlete_df = df.drop_duplicates(subset=['Name', 'region'])
    athlete_df.fillna({'Medal':'No Medal'}, inplace=True)
    if sport != 'Overall':
        temp_df = athlete_df[athlete_df['Sport'] == sport]
        return temp_df
    else:
        return athlete_df

def men_vs_women(df):
    athlete_df = df.drop_duplicates(subset=['Name', 'region'])

    men = athlete_df[athlete_df['Sex'] == 'M'].groupby('Year').count()['Name'].reset_index()
    women = athlete_df[athlete_df['Sex'] == 'F'].groupby('Year').count()['Name'].reset_index()

    final = men.merge(women, on='Year', how='left')
    final.rename(columns={'Name_x': 'Male', 'Name_y': 'Female'}, inplace=True)

    final.fillna(0, inplace=True)

    return final

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff


%matplotlib inline

In [ ]:

df = pd.read_csv('athlete_events.csv')
region_df = pd.read_csv('noc_regions.csv')

df = preprocess(df, region_df)
df.head()

In [ ]:

print("Total Editipns:", df['Year'].nunique() - 1)
print("Total host cities:", df['City'].nunique())
print("total sports:", df['Sport'].nunique())
print("Total events:", df['Event'].nunique())
print("Total athletes:", df['Name'].nunique())
print("Total nations:", df['region'].nunique())

In [ ]:
nations_over_time = data_over_time(df, 'region')

px.line(nations_over_time,
        x = "Edition",
        y = "region",
        title = "Participating nations over years")

In [ ]:

events_over_time = data_over_time(df, 'Event')
px.line(events_over_time,
        x = "Edition",
        y = "Event",
        title = "Events over years")

In [ ]:

plt.figure(figsize = (20,20))
x = df.drop_duplicates(['Year', 'Sport', 'Event'])
sns.heatmap(
    x.pivot_table(index='Sport',
                  columns='Year',
                  values='Event',
                  aggfunc='count').fillna(0).astype(int),
    annot=False
)
plt.title("Number of events per sport over time")
plt.show()

In [ ]:

selected_country = 'India'
country_df =  yearwise_medal_tally(df, selected_country)
px.line(country_df,
        x="Year",
        y="Medal",
        title="India medal tally over the years")

In [ ]:

 most_successful_countrywise(df, selected_country)

In [ ]:

athlete_df = df.drop_duplicates(['Name','region'])
x1 = athlete_df['Age'].dropna()
x2 = athlete_df[athlete_df['Medal']=='Gold']['Age'].dropna()
fig = ff.create_distplot(
    [x1,x2],
    ['Overall athletes', 'Gold medalists'],
    show_hist=True)
fig

In [ ]:
temp_df =  weight_v_height(df, 'Overall')

plt.figure(figsize=(8,6))
sns.scatterplot(x=temp_df['Weight'],
                y=temp_df['Height'],
                hue=temp_df['Medal'],
                style=temp_df['Sex'])
plt.title("Height vs Weight Distribution")
plt.show()

In [ ]:
final =  men_vs_women(df)

px.line(final,
        x="Year",
        y=["Male","Female"],
        title="Male vs Female Participation Over Years")

# Conclusion
From the analysis:

- Olympic participation has increased significantly over time.
- The number of sports and events has grown steadily.
- Some countries dominate medal tallies consistently.
- Athlete demographics vary across sports.
- Female participation has increased considerably in recent decades.
  
This analysis demonstrates trends, performance patterns, and demographic insights in Olympic history.